# Работа через API

In [1]:
import requests
import pandas as pd

In [2]:
API_KEY = '64EMAA5-XMT49EZ-Q6GWVPP-NDZTQAA'
BASE_URL = 'https://api.poiskkino.dev/v1/movie'
HEADERS = {'X-API-KEY': API_KEY}

In [3]:
result_list = {'title': [], 'genre': [], 'year': [],
               'country': [], 'rate': [], 'description': []
              }
params = {'limit': 250,
          'selectFields': ['name', 'genres', 'year', 'countries', 'rating', 'description']
         }

try:
    response = requests.get(BASE_URL, headers=HEADERS, params=params)
    response.raise_for_status()
    data = response.json()

    for movie in data.get('docs', []):
        result_list['title'].append(movie.get('name', '—'))
        result_list['year'].append(movie.get('year', '—'))
        result_list['description'].append(movie.get('description', '—'))
        
        genres = [g.get('name') for g in movie.get('genres', [])]
        result_list['genre'].append(", ".join(genres) if genres else "—")
        
        countries = [c.get('name') for c in movie.get('countries', [])]
        result_list['country'].append(", ".join(countries) if countries else "—")
        
        rate = movie.get('rating', {}).get('kp', 0.0)
        result_list['rate'].append(rate)

    df = pd.DataFrame(result_list)

except Exception as e:
    print(f"Ошибка запроса: {e}")

In [4]:
for key, value in result_list.items():
    print(f"{key}: {len(value)}")

title: 250
genre: 250
year: 250
country: 250
rate: 250
description: 250


In [5]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
genre - 0
year - 0
country - 0
rate - 0
description - 0


In [6]:
df.to_csv('DataTop250_API.csv', index=False)
df.head(10)

,title,genre,year,country,rate,description
0,1+1,"драма, комедия",2011,Франция,8.862,"Пострадав в результате несчастного случая, бог..."
1,Джентльмены,"криминал, комедия, боевик",2019,"США, Великобритания",8.682,Один ушлый американец ещё со студенческих лет ...
2,Триггер,драма,2018,Россия,8.451,Психолог Артём Стрелецкий — сторонник шоковой ...
3,Гнев человеческий,"боевик, триллер",2021,"Великобритания, США",7.723,Грузовики лос-анджелесской инкассаторской комп...
4,Брат,"драма, криминал, боевик",1997,Россия,8.375,"Демобилизовавшись, Данила Багров возвращается ..."
5,Волк с Уолл-стрит,"драма, криминал, биография, комедия",2013,США,8.106,1987 год. Джордан Белфорт становится брокером ...
6,Фишер,"детектив, драма, криминал, триллер",2023,Россия,7.851,"Интеллигентный следователь Валерий Козырев, ег..."
7,Игра престолов,"фэнтези, драма, боевик, мелодрама, приключения",2011,"США, Великобритания",9.015,"К концу подходит время благоденствия, и лето, ..."
8,Телохранители,комедия,2023,Россия,8.269,Три боксера из Челябинска перед вылетом на Чем...
9,Достать ножи,"детектив, комедия, драма, криминал",2019,США,8.193,На следующее утро после празднования 85-летия ...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        250 non-null    object 
 1   genre        250 non-null    object 
 2   year         250 non-null    int64  
 3   country      250 non-null    object 
 4   rate         250 non-null    float64
 5   description  250 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 11.8+ KB


# Объединение файлов (создание датасета из 500 фильмов)

In [8]:
file_html = 'Top250Films.csv'
file_api = 'DataTop250_API.csv'

df1 = pd.read_csv(file_html)
df2 = pd.read_csv(file_api)

df_combined = pd.concat([df2, df1], ignore_index=True)

df_combined.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)

df_combined['rate'] = pd.to_numeric(df_combined['rate'], errors='coerce').fillna(0.0)
df_combined['rate'] = df_combined['rate'].round(1)

df_combined.drop_duplicates(subset=['title', 'year'], keep='first', inplace=True)
df_combined.dropna(subset=['title'], inplace=True)

df_combined.fillna({'genre': '—', 'country': '—', 'description': 'Нет описания'}, inplace=True)

output_file = 'Final_Movies_Dataset.csv'
df_combined.to_csv(output_file, index=False)

print(f"Всего записей в финальном файле: {len(df_combined)}")

Всего записей в финальном файле: 497


In [9]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   247 non-null    int64  
 1   title        247 non-null    object 
 2   genre        247 non-null    object 
 3   year         247 non-null    object 
 4   country      247 non-null    object 
 5   rate         247 non-null    float64
 6   description  247 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 13.6+ KB


In [10]:
df1['rate'] = pd.to_numeric(df['rate'], errors='coerce')
display(df1.nlargest(10, 'rate')[['title', 'rate', 'year', 'country']])

,title,rate,year,country
73,Собиратель душ,9.196,2024,Канада
37,Уроки фарси,9.111,2020,Россия
41,Конец славы,9.079,2024,Россия
7,Уволить Жору (2026),9.015,2026,Россия
135,Один день в Стамбуле,8.982,2024,Россия
60,Нэчжа побеждает Царя драконов,8.929,2025,Китай
90,Хочу замуж,8.929,2022,Россия
54,Финист. Первый богатырь,8.920,2025,Россия
131,Наследник (2023),8.890,2023,Южная Корея
0,Ангелы Ладоги,8.862,2026,Россия


In [11]:
df1['country'].describe()

count        247
unique        20
top       Россия
freq         156
Name: country, dtype: object

In [12]:
df1['genre'].describe()

count         247
unique        108
top       Комедия
freq           37
Name: genre, dtype: object

In [13]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        250 non-null    object 
 1   genre        250 non-null    object 
 2   year         250 non-null    int64  
 3   country      250 non-null    object 
 4   rate         250 non-null    float64
 5   description  250 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 11.8+ KB


In [14]:
df2['rate'] = pd.to_numeric(df['rate'], errors='coerce')
display(df2.nlargest(10, 'rate')[['title', 'rate', 'year', 'country']])

,title,rate,year,country
73,Жил-был пёс,9.196,1982,СССР
37,Побег из Шоушенка,9.111,1994,США
41,Зеленая миля,9.079,1999,США
7,Игра престолов,9.015,2011,"США, Великобритания"
135,Гравити Фолз,8.982,2012,"США, Корея Южная"
60,Рик и Морти,8.929,2013,США
90,Во все тяжкие,8.929,2008,США
54,Форрест Гамп,8.920,1994,США
131,Молодёжка. Новая смена,8.890,2024,Россия
0,1+1,8.862,2011,Франция


In [15]:
df2['country'].describe()

count        250
unique        48
top       Россия
freq          76
Name: country, dtype: object

In [16]:
df2['genre'].describe()

count         250
unique        168
top       комедия
freq           10
Name: genre, dtype: object

In [17]:
df3 = pd.read_csv('All_Dataset.csv')
df3.info()

FileNotFoundError: [Errno 2] No such file or directory: 'All_Dataset.csv'